# Auto-encodeurs débruiteurs et variationnels avec Keras

Le notebook précédent entraînait des auto-encodeurs à reconstruire leur entrée. Celui-ci change d'abord la tâche : retrouver une image propre à partir d'une version bruitée. Puis il change le modèle, pour pouvoir générer de nouvelles images. Les données sont celles de Fashion-MNIST : 60 000 images d'entraînement et 10 000 de test, 28 × 28 en niveaux de gris comme MNIST, mais de 10 sortes de vêtements et d'accessoires.

1. Corrompre les images
2. Un débruiteur dense
3. Un code sur-complet
4. Un débruiteur convolutif
5. Un auto-encodeur variationnel
6. Générer des images
7. L'effet de β

Chaque partie réutilise les données et les modèles des précédentes. Pas besoin de GPU : le notebook tourne en une vingtaine de minutes à une demi-heure sur un processeur, dont un quart pour le débruiteur convolutif de la partie 4. Un GPU (`Exécution > Modifier le type d'exécution`) l'accélère nettement.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers, ops

keras.utils.set_random_seed(0)

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
noms_classes = [
  "T-shirt",
  "Pantalon",
  "Pull",
  "Robe",
  "Manteau",
  "Sandale",
  "Chemise",
  "Basket",
  "Sac",
  "Bottine",
]
dim_entree = 28 * 28
# Pixels dans [0, 1], comme la sortie d'une sigmoïde
X_train = X_train.reshape(-1, dim_entree) / 255
X_test = X_test.reshape(-1, dim_entree) / 255
print(X_train.shape, X_test.shape)


def affiche_images(
  *rangees: np.ndarray, titres: list | np.ndarray | None = None
) -> None:
  # Affiche des rangées d'images 28 × 28, une rangée par tableau
  n = len(rangees[0])
  _, axes = plt.subplots(
    len(rangees), n, figsize=(n * 1.2, len(rangees) * 1.3), squeeze=False
  )
  for rangee, images in zip(axes, rangees):
    for ax, image in zip(rangee, images):
      ax.imshow(np.reshape(image, (28, 28)), cmap="gray_r")
      ax.axis("off")
  for ax, titre in zip(axes[0], [] if titres is None else titres):
    ax.set_title(titre, fontsize=8)
  plt.show()


affiche_images(X_train[:10], titres=[noms_classes[y] for y in y_train[:10]])

## 1. Corrompre les images

Un débruiteur apprend à retrouver l'image propre à partir d'une version abîmée : ses exemples d'entraînement sont des paires (image bruitée, image propre). Fabriquez les images bruitées :

- `X_train_bruite` et `X_test_bruite` : `X_train` et `X_test`, plus un bruit gaussien de moyenne 0 et d'écart-type 0,5 tiré indépendamment pour chaque pixel ;
- ramenez ensuite chaque pixel dans [0, 1], comme ceux d'une vraie image.

La cellule après la solution affiche des images bruitées au-dessus des images propres.

**Documentation utile**

- [`numpy.random.Generator.normal`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html)
- [`numpy.clip`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `rng = np.random.default_rng(0)`, puis `rng.normal(0, 0.5, size=X_train.shape)` : un bruit de la forme des images, un tirage par pixel
- `np.clip(X, 0, 1)` : les valeurs sous 0 deviennent 0, celles au-dessus de 1 deviennent 1
- vérification : `X_train_bruite.min()` vaut 0, `X_train_bruite.max()` vaut 1, et `X_train_bruite.shape` est celle de `X_train`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rng = np.random.default_rng(0)
X_train_bruite = np.clip(X_train + rng.normal(...), ..., ...)
X_test_bruite = ...
```

### Solution

In [ ]:
rng = np.random.default_rng(0)
X_train_bruite = np.clip(X_train + rng.normal(0, 0.5, size=X_train.shape), 0, 1)
X_test_bruite = np.clip(X_test + rng.normal(0, 0.5, size=X_test.shape), 0, 1)

### Les images bruitées

In [ ]:
titres = [noms_classes[y] for y in y_test[:10]]
affiche_images(X_test_bruite[:10], X_test[:10], titres=titres)

## 2. Un débruiteur dense

Construisez `autoencodeur`, un débruiteur dense à une couche cachée :

- une couche `Dense` de `dim_code = 12` unités et d'activation ReLU, le code, sur des entrées de taille `dim_entree` ;
- la couche de sortie, de `dim_entree` unités, avec une activation adaptée à des pixels dans [0, 1] ;
- compilé avec l'optimiseur Adam et la perte `"mse"`.

La cellule après la solution l'entraîne sur les paires (image bruitée, image propre), puis affiche des images de test bruitées, leur version débruitée et l'image propre.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- un seul `keras.Sequential` suffit : `keras.Input(shape=(dim_entree,))`, puis les deux couches `Dense`
- la sortie : `activation="sigmoid"`, qui rend des valeurs dans ]0, 1[, comme les pixels
- `autoencodeur.summary()` : 9 420 paramètres pour la couche du code (784 × 12 + 12) et 10 192 pour la sortie

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
dim_code = 12
autoencodeur = keras.Sequential(
  [keras.Input(shape=(dim_entree,)), layers.Dense(...), layers.Dense(...)],
  name="autoencodeur",
)
autoencodeur.compile(...)
autoencodeur.summary()
```

### Solution

In [ ]:
dim_code = 12
autoencodeur = keras.Sequential(
  [
    keras.Input(shape=(dim_entree,)),
    layers.Dense(dim_code, activation="relu"),
    layers.Dense(dim_entree, activation="sigmoid"),
  ],
  name="autoencodeur",
)
autoencodeur.compile(optimizer="adam", loss="mse")
autoencodeur.summary()

### Entraînement

L'entrée est l'image bruitée, la cible l'image propre. En haut, les images bruitées, puis leur version débruitée, puis les images propres.

In [ ]:
autoencodeur.fit(
  X_train_bruite,
  X_train,
  epochs=20,
  batch_size=256,
  validation_data=(X_test_bruite, X_test),
  verbose=2,
)
X_test_debruite = autoencodeur.predict(X_test_bruite, verbose=0)
affiche_images(X_test_bruite[:10], X_test_debruite[:10], X_test[:10], titres=titres)

## 3. Un code sur-complet

Un code de 12 nombres limite ce que le débruiteur peut restituer. Avec un code aussi grand que l'entrée, ou plus, un auto-encodeur ordinaire n'a plus rien à résumer : il peut se contenter de recopier son entrée (slide « Code sous-complet ou sur-complet »). Un débruiteur le fait-il aussi ? Comparez-les :

- écrivez `construit_autoencodeur_dense(dim_code)`, qui rend le modèle de la partie 2, compilé de même, pour une taille de code donnée ;
- entraînez-en deux à code de 1 024 unités, plus que les 784 pixels, pendant 10 epochs chacun : `ordinaire`, un auto-encodeur ordinaire, sur les paires (`X_train`, `X_train`), et `debruiteur`, un débruiteur, sur les paires (`X_train_bruite`, `X_train`) ;
- appliquez-les tous deux aux images de test bruitées, `X_test_bruite`, donnez l'erreur quadratique moyenne de chacun par rapport aux images propres, et affichez les images bruitées, la sortie de `ordinaire`, celle de `debruiteur`, puis les images propres.

Lequel recopie le bruit ? Pourquoi l'autre ne le fait-il pas, alors qu'il en aurait la place ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- le corps de `construit_autoencodeur_dense` est la solution de la partie 2, avec `dim_code` en paramètre, suivie de `return autoencodeur` ; `construit_autoencodeur_dense(1024).summary()` : 1 607 440 paramètres (784 × 1 024 + 1 024, puis 1 024 × 784 + 784)
- `ordinaire.fit(X_train, X_train, epochs=10, batch_size=256, verbose=0)` : la cible est l'entrée ; pour `debruiteur`, l'entrée est `X_train_bruite`, la cible `X_train`. Chacun prend quelques minutes
- `X_test_ordinaire = ordinaire.predict(X_test_bruite, verbose=0)`, puis `np.mean((X_test_ordinaire - X_test) ** 2)` : l'erreur par rapport aux images propres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def construit_autoencodeur_dense(dim_code: int) -> keras.Model:
  autoencodeur = ...
  autoencodeur.compile(...)
  return autoencodeur


ordinaire = construit_autoencodeur_dense(1024)
ordinaire.fit(...)
debruiteur = ...
debruiteur.fit(...)
X_test_ordinaire = ordinaire.predict(X_test_bruite, verbose=0)
X_test_debruite = ...
print(f"Erreurs : ordinaire {...:.4f}, débruiteur {...:.4f}")
affiche_images(X_test_bruite[:10], ..., ..., X_test[:10], titres=titres)
```

### Solution

In [ ]:
def construit_autoencodeur_dense(dim_code: int) -> keras.Model:
  autoencodeur = keras.Sequential(
    [
      keras.Input(shape=(dim_entree,)),
      layers.Dense(dim_code, activation="relu"),
      layers.Dense(dim_entree, activation="sigmoid"),
    ],
    name="autoencodeur",
  )
  autoencodeur.compile(optimizer="adam", loss="mse")
  return autoencodeur


ordinaire = construit_autoencodeur_dense(1024)  # plus que les 784 pixels
ordinaire.fit(X_train, X_train, epochs=10, batch_size=256, verbose=0)
debruiteur = construit_autoencodeur_dense(1024)
debruiteur.fit(X_train_bruite, X_train, epochs=10, batch_size=256, verbose=0)
X_test_ordinaire = ordinaire.predict(X_test_bruite, verbose=0)
X_test_debruite = debruiteur.predict(X_test_bruite, verbose=0)
print(
  f"Auto-encodeur ordinaire : erreur {np.mean((X_test_ordinaire - X_test) ** 2):.4f}"
)
print(f"Débruiteur : erreur {np.mean((X_test_debruite - X_test) ** 2):.4f}")
# En haut les images bruitées, puis l'AE ordinaire, le débruiteur, les images propres
affiche_images(
  X_test_bruite[:10],
  X_test_ordinaire[:10],
  X_test_debruite[:10],
  X_test[:10],
  titres=titres,
)

L'auto-encodeur ordinaire recopie le bruit : il n'a appris qu'à rendre son entrée, et son code, plus large que l'image, lui permet de la rendre presque telle quelle, bruit compris. Son erreur par rapport aux images propres est donc bien plus forte. Le débruiteur, avec le même code, rend des images propres et plus nettes que celui de la partie 2 : sa cible est l'image propre, et chaque pixel de bruit recopié augmenterait sa perte. C'est ce qui permet à un débruiteur d'avoir un code sur-complet sans apprendre l'identité, contrairement à un auto-encodeur ordinaire.

## 4. Un débruiteur convolutif

Une couche dense traite chaque pixel à part ; une convolution exploite le voisinage de chaque pixel, ce qui convient mieux à une image. Les couches de convolution attendent une image 28 × 28 × 1, avec un axe pour les canaux (un seul en niveaux de gris, trois en couleur) : la cellule suivante remet les images dans cette forme.

In [ ]:
X_train_img = X_train.reshape(-1, 28, 28, 1)
X_test_img = X_test.reshape(-1, 28, 28, 1)
X_train_bruite_img = X_train_bruite.reshape(-1, 28, 28, 1)
X_test_bruite_img = X_test_bruite.reshape(-1, 28, 28, 1)
print(X_train_img.shape)

### Le modèle

Construisez un `encodeur` et un `decodeur` entièrement convolutifs, puis `autoencodeur`, qui les enchaîne. Toutes les `Conv2D` ont des filtres 3 × 3 et `padding="same"` :

- `encodeur`, sur des entrées 28 × 28 × 1 : une `Conv2D` de 16 filtres (ReLU) et un `MaxPooling2D` 2 × 2, puis une `Conv2D` de 8 filtres (ReLU) et un `MaxPooling2D` 2 × 2. Le code est le tableau 7 × 7 × 8 qui en sort ;
- `decodeur`, le chemin inverse : une `Conv2D` de 8 filtres (ReLU) et un `UpSampling2D` 2 × 2, une `Conv2D` de 16 filtres (ReLU) et un `UpSampling2D` 2 × 2, puis une dernière `Conv2D` à 1 filtre et activation sigmoïde, qui rend une image 28 × 28 × 1 ;
- `autoencodeur`, compilé avec l'optimiseur Adam et la perte `"mse"`, comme les débruiteurs denses, pour que leurs erreurs se comparent.

Vérifiez les formes avec `autoencodeur.summary(expand_nested=True)`, et comparez son nombre de paramètres à celui des débruiteurs denses. Pourquoi ne pas finir le décodeur par l'`UpSampling2D` ?

**Documentation utile**

- [`keras.layers.UpSampling2D`](https://keras.io/api/layers/reshaping_layers/up_sampling2d/)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `layers.Conv2D(16, 3, activation="relu", padding="same")` : `padding="same"` garde la taille de l'image ; une petite fonction `cree_convolution(filtres, activation="relu")` qui la renvoie évite de répéter ces arguments
- `layers.MaxPooling2D(2)` divise la hauteur et la largeur par 2 (28 → 14 → 7), `layers.UpSampling2D(2)` les multiplie par 2 en répétant chaque pixel (7 → 14 → 28)
- le décodeur prend des entrées `keras.Input(shape=(7, 7, 8))`, la forme du code
- la dernière `Conv2D` : `cree_convolution(1, "sigmoid")` ; un `UpSampling2D` en fin de réseau rendrait des blocs de 2 × 2 pixels identiques, et une sortie sans sigmoïde pourrait sortir de [0, 1]
- `summary` : la sortie de l'encodeur est `(None, 7, 7, 8)`, celle du décodeur `(None, 28, 28, 1)` ; 3 217 paramètres en tout, contre 9 420 + 10 192 = 19 612 pour le débruiteur dense à code de 12 et 1 607 440 pour ceux de la partie 3 : chaque filtre 3 × 3 sert en tout point de l'image

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def cree_convolution(filtres: int, activation: str = "relu") -> layers.Conv2D:
  return layers.Conv2D(filtres, 3, activation=activation, padding="same")


encodeur = keras.Sequential(
  [
    keras.Input(shape=(28, 28, 1)),
    cree_convolution(16),
    layers.MaxPooling2D(2),
    ...,
    ...,
  ],
  name="encodeur",
)
decodeur = keras.Sequential(
  [
    keras.Input(shape=(7, 7, 8)),
    cree_convolution(8),
    layers.UpSampling2D(2),
    ...,
    ...,
    ...,
  ],
  name="decodeur",
)
autoencodeur = keras.Sequential([encodeur, decodeur], name="autoencodeur")
autoencodeur.compile(...)
autoencodeur.summary(expand_nested=True)
```

#### Solution

In [ ]:
def cree_convolution(filtres: int, activation: str = "relu") -> layers.Conv2D:
  return layers.Conv2D(filtres, 3, activation=activation, padding="same")


encodeur = keras.Sequential(
  [
    keras.Input(shape=(28, 28, 1)),
    cree_convolution(16),
    layers.MaxPooling2D(2),  # 14 × 14 × 16
    cree_convolution(8),
    layers.MaxPooling2D(2),  # 7 × 7 × 8, le code
  ],
  name="encodeur",
)
decodeur = keras.Sequential(
  [
    keras.Input(shape=(7, 7, 8)),
    cree_convolution(8),
    layers.UpSampling2D(2),  # 14 × 14 × 8
    cree_convolution(16),
    layers.UpSampling2D(2),  # 28 × 28 × 16
    # Lisse les blocs de l'UpSampling2D, sortie dans ]0, 1[
    cree_convolution(1, "sigmoid"),
  ],
  name="decodeur",
)
autoencodeur = keras.Sequential([encodeur, decodeur], name="autoencodeur")
autoencodeur.compile(optimizer="adam", loss="mse")  # comme les débruiteurs denses
autoencodeur.summary(expand_nested=True)

### Entraînement

5 epochs seulement, contre 20 et 10 pour les débruiteurs denses : chacune est bien plus longue. La perte est la même, la MSE : les erreurs se comparent.

In [ ]:
# 5 epochs : chacune prend bien plus longtemps que celle d'un débruiteur dense
autoencodeur.fit(
  X_train_bruite_img,
  X_train_img,
  epochs=5,
  batch_size=256,
  validation_data=(X_test_bruite_img, X_test_img),
  verbose=2,
)
X_test_debruite = autoencodeur.predict(X_test_bruite_img, verbose=0)
print(f"Erreur : {np.mean((X_test_debruite - X_test_img) ** 2):.4f}")
affiche_images(X_test_bruite[:10], X_test_debruite[:10], X_test[:10], titres=titres)

### Sur des images non bruitées

Le débruiteur n'a vu que des entrées bruitées. Que rend-il d'une image propre ?

In [ ]:
X_test_reconstruit = autoencodeur.predict(X_test_img[:10], verbose=0)
affiche_images(X_test[:10], X_test_reconstruit, titres=titres)

Sur une image propre, le débruiteur rend la forme du vêtement, mais lissée : il efface les détails fins, comme les logos, les carreaux ou les rayures, et force le contraste. Il a appris à garder ce qui ressemble à un vêtement et à effacer le reste ; un détail fin, à ses yeux, ressemble à du bruit. Son erreur sur les images bruitées se place entre celles du débruiteur dense à code de 12 et du débruiteur sur-complet, avec 3 217 paramètres au lieu de 19 612 et 1 607 440, et en 5 epochs : chaque filtre 3 × 3 sert en tout point de l'image, quand chaque poids d'une couche dense ne relie qu'un pixel à une unité.

## 5. Un auto-encodeur variationnel

Le notebook précédent l'a montré : décoder un code tiré au hasard ne donne pas une image vraisemblable, car les codes d'un auto-encodeur n'occupent qu'une petite partie de l'espace latent, sans forme connue. Un auto-encodeur variationnel (VAE) encode chaque image en une loi, $\mathcal{N}(\mu, \sigma^2)$, et une perte KL pousse ces lois vers $\mathcal{N}(0, I)$ : on peut alors tirer un code dans $\mathcal{N}(0, I)$ et le décoder.

Ce qui est propre au VAE tient dans une couche, `Echantillonnage` (slides « Astuce de reparamétrisation » et « Perte d'un VAE ») : écrivez sa méthode `call`, son `__init__` est fourni. Elle reçoit `[z_moyenne, z_log_var]`, $\mu$ et $\log \sigma^2$, deux tenseurs de forme (taille du lot, `dim_latente`), et doit :

- calculer la KL de chaque exemple, $-\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$, et l'ajouter à la perte du modèle avec `self.add_loss`, en moyenne sur le lot et multipliée par `self.beta` ;
- tirer $\varepsilon \sim \mathcal{N}(0, I)$, de la forme de `z_moyenne`, avec le générateur `self.generateur_graine`, et rendre $z = \mu + \sigma \varepsilon$.

`beta` vaut 1 / 784 par défaut : la perte `"binary_crossentropy"` du modèle est une moyenne sur les 784 pixels, alors que la KL est une somme sur les dimensions du code. Les cellules après la solution assemblent le VAE autour de `Echantillonnage` et l'entraînent.

**Documentation utile**

- [`Layer.add_loss`](https://keras.io/api/layers/base_layer/#addloss-method)
- [`keras.random.normal`](https://keras.io/api/random/random_ops/#normal-function)
- [`keras.ops.sum`](https://keras.io/api/ops/numpy/#sum-function)
- [`keras.ops.mean`](https://keras.io/api/ops/numpy/#mean-function)
- [`keras.ops.square`](https://keras.io/api/ops/numpy/#square-function)
- [`keras.ops.exp`](https://keras.io/api/ops/numpy/#exp-function)

In [ ]:
class Echantillonnage(layers.Layer):
  def __init__(self, beta: float = 1 / 784, **kwargs) -> None:
    super().__init__(**kwargs)
    self.beta = beta  # poids de la KL
    self.generateur_graine = keras.random.SeedGenerator(0)

  def call(self, entrees):
    z_moyenne, z_log_var = entrees
    # Votre code ici
    ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `ops.exp(z_log_var)` : $\sigma^2$ ; `ops.exp(0.5 * z_log_var)` : $\sigma$ ; `ops.square(z_moyenne)` : $\mu^2$
- `ops.sum(..., axis=1)` : la somme sur les dimensions du code, une KL par exemple ; `ops.mean(kl)` : sa moyenne sur le lot
- `self.add_loss(self.beta * ops.mean(kl))` : Keras ajoute ce terme à la perte du modèle, à chaque pas d'entraînement
- `keras.random.normal(ops.shape(z_moyenne), seed=self.generateur_graine)` : $\varepsilon$, un nouveau tirage à chaque appel
- vérification : `s = Echantillonnage(beta=1.0)`, puis `s([ops.array([[1.0, 0.0]]), ops.zeros((1, 2))])` ; `float(s.losses[0])` vaut alors 0,5 (μ = (1, 0) et σ = 1 : la KL vaut μ₁² / 2)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class Echantillonnage(layers.Layer):
  def __init__(self, beta: float = 1 / 784, **kwargs) -> None:
    super().__init__(**kwargs)
    self.beta = beta  # poids de la KL
    self.generateur_graine = keras.random.SeedGenerator(0)

  def call(self, entrees):
    z_moyenne, z_log_var = entrees
    kl = -0.5 * ops.sum(..., axis=1)
    self.add_loss(...)
    eps = keras.random.normal(...)
    return ...
```

### Solution

In [ ]:
class Echantillonnage(layers.Layer):
  def __init__(self, beta: float = 1 / 784, **kwargs) -> None:
    super().__init__(**kwargs)
    self.beta = beta  # poids de la KL
    self.generateur_graine = keras.random.SeedGenerator(0)

  def call(self, entrees):
    z_moyenne, z_log_var = entrees
    kl = -0.5 * ops.sum(
      1 + z_log_var - ops.square(z_moyenne) - ops.exp(z_log_var), axis=1
    )
    self.add_loss(self.beta * ops.mean(kl))  # + perte du modèle
    eps = keras.random.normal(ops.shape(z_moyenne), seed=self.generateur_graine)
    return z_moyenne + ops.exp(0.5 * z_log_var) * eps  # z = μ + σ·ε

### Le modèle

`construit_vae(beta)` assemble le VAE autour de `Echantillonnage`, avec un espace latent à `dim_latente = 2` dimensions, pour pouvoir le dessiner :

- l'encodeur, en API fonctionnelle : une entrée de taille `dim_entree`, une `Dense` de 256 unités (ReLU), puis deux `Dense` de `dim_latente` unités sans activation, `z_moyenne` et `z_log_var`, et `z`, la sortie de `Echantillonnage(beta)` sur les deux ; `encodeur` est le `keras.Model` qui rend `[z_moyenne, z_log_var, z]` ;
- `decodeur`, un `keras.Sequential` : une `Dense` de 256 unités (ReLU), puis la sortie de `dim_entree` unités en sigmoïde ;
- `vae`, le `keras.Model` de l'entrée à `decodeur(z)`, compilé avec Adam et la perte `"binary_crossentropy"`, à laquelle `Echantillonnage` ajoute la KL.

`vae.summary()` compte 404 244 paramètres : `Echantillonnage` n'en a aucun.

In [ ]:
dim_latente = 2


def construit_vae(
  beta: float = 1 / 784,
) -> tuple[keras.Model, keras.Model, keras.Model]:
  entrees = keras.Input(shape=(dim_entree,))
  h = layers.Dense(256, activation="relu")(entrees)
  z_moyenne = layers.Dense(dim_latente)(h)
  z_log_var = layers.Dense(dim_latente)(h)  # log σ², un réel
  z = Echantillonnage(beta)([z_moyenne, z_log_var])
  encodeur = keras.Model(entrees, [z_moyenne, z_log_var, z], name="encodeur")
  decodeur = keras.Sequential(
    [
      keras.Input(shape=(dim_latente,)),
      layers.Dense(256, activation="relu"),
      layers.Dense(dim_entree, activation="sigmoid"),
    ],
    name="decodeur",
  )
  vae = keras.Model(entrees, decodeur(z), name="vae")
  vae.compile(optimizer="adam", loss="binary_crossentropy")  # + KL
  return encodeur, decodeur, vae


encodeur, decodeur, vae = construit_vae()
vae.summary()

### Entraînement et espace latent

La cible est l'image propre elle-même. Chaque point du nuage est le $\mu$ d'une image de test, coloré selon sa classe.

In [ ]:
vae.fit(
  X_train,
  X_train,
  epochs=20,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=2,
)
moyennes_test, _, _ = encodeur.predict(X_test, verbose=0)
plt.figure(figsize=(7, 6))
for etiquette, nom in enumerate(noms_classes):
  points = moyennes_test[y_test == etiquette]
  plt.scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, label=nom)
plt.legend(markerscale=5, fontsize=8)
plt.xlabel("$z_1$")
plt.ylabel("$z_2$")
plt.show()

Les μ des images de test restent presque tous entre -3 et 3 sur chaque axe, autour de l'origine : la KL tire chaque loi vers $\mathcal{N}(0, I)$. Chaque classe occupe sa région, sans jamais avoir vu d'étiquette. Les chaussures (sandales, baskets, bottines) forment un groupe à part, séparé des vêtements par une bande presque vide ; les pantalons et les sacs ont chacun leur région. Les hauts (pulls, manteaux, chemises) se chevauchent : ils se ressemblent, et deux dimensions ne suffisent pas à les séparer. L'emplacement de chaque groupe change d'un entraînement à l'autre : les axes de l'espace latent n'ont pas de sens fixé.

## 6. Générer des images

Un VAE est un modèle génératif : un code tiré dans $\mathcal{N}(0, I)$, puis décodé, doit donner une image vraisemblable. La cellule suivante tire 16 codes, `z_nouveau`, et les décode avec `decodeur`. Les images ressemblent-elles à des vêtements, contrairement à celles décodées depuis du bruit dans le notebook précédent ?

In [ ]:
z_nouveau = keras.random.normal((16, dim_latente), seed=0)  # z ~ N(0, I)
generees = decodeur.predict(z_nouveau, verbose=0)
affiche_images(generees)

La plupart ressemblent à des vêtements : des hauts, des pantalons, des chaussures, flous comme toutes les images d'un VAE à deux dimensions. Quelques-unes, pâles, mélangent deux sortes de vêtements : leur code est tombé entre deux régions, où le décodeur hésite. Dans le notebook précédent, un code tiré au hasard ne donnait que des taches : ici, la KL a placé les codes des images d'entraînement là où $\mathcal{N}(0, I)$ tire les siens, et le décodeur sait les décoder. Un espace latent plus grand donnerait des images plus nettes, mais on ne pourrait plus en dessiner la carte.

### La carte de l'espace latent

Avec 2 dimensions, on peut décoder une grille régulière de codes, de -2 à 2 sur chaque axe, là où $\mathcal{N}(0, I)$ tire presque tous ses points : chaque case montre ce que le décodeur rend en ce point. Comparez avec le nuage de la partie 5.

In [ ]:
n = 12
grille = np.linspace(-2, 2, n)
# z_2 décroît de haut en bas, comme l'axe vertical du nuage
codes = np.array([[z1, z2] for z2 in grille[::-1] for z1 in grille])
images = decodeur.predict(codes, verbose=0).reshape(n, n, 28, 28)
plt.figure(figsize=(8, 8))
plt.imshow(images.transpose(0, 2, 1, 3).reshape(n * 28, n * 28), cmap="gray_r")
plt.axis("off")
plt.show()

La carte reprend le nuage de la partie 5 : chaque sorte de vêtement occupe la région de ses μ, les chaussures à part. D'une case à sa voisine, l'image change peu : chaque point de l'espace latent se décode en une image, et des codes proches en images proches. Entre deux régions, les images sont pâles et mélangées.

## 7. L'effet de β

Le poids `beta` de la KL règle un compromis (slide « Perte d'un VAE ») : sans KL, rien ne rapproche les lois de $\mathcal{N}(0, I)$ ; plus la KL pèse, plus les lois se confondent avec $\mathcal{N}(0, I)$, et moins le code dit de l'image. Observez-le :

- avec `construit_vae`, entraînez deux autres VAE, l'un avec `beta=0`, l'autre avec un β dix fois plus grand que celui de la partie 5, `beta=10 / 784`, pendant 20 epochs chacun, avec le même `fit` (et `verbose=0`) ;
- pour chacun des trois β, avec le VAE de la partie 5, tracez côte à côte les nuages des μ des images de test, colorés selon leur classe, et donnez le σ moyen de leurs lois et l'erreur quadratique moyenne de reconstruction de leurs μ décodés ;
- affichez, pour chaque β, les images que son décodeur rend des 16 codes `z_nouveau` de la partie 6.

Regardez l'échelle des axes. Où sont les μ sans KL, et que vaut σ ? Les images générées ressemblent-elles encore à des vêtements ? Que gagne-t-on, et que perd-on, avec un β dix fois plus grand ?

**Documentation utile**

- [`matplotlib.pyplot.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `encodeur_beta, decodeur_beta, vae_beta = construit_vae(0)`, puis `vae_beta.fit(X_train, X_train, epochs=20, batch_size=256, verbose=0)` : quelques minutes par VAE sur un processeur
- un dictionnaire `modeles = {1 / 784: (encodeur, decodeur)}`, complété des deux autres : `sorted(modeles.items())` le parcourt par β croissant
- `_, axes = plt.subplots(1, 3, figsize=(18, 5))` : trois graphiques côte à côte, où `ax.scatter` et `ax.set_title` remplacent `plt.scatter` et `plt.title`
- `encodeur_beta(X_test)` appelle l'encodeur directement et rend ses trois sorties, que `ops.convert_to_numpy` change en tableaux NumPy : les μ, les log σ² et les z ; `np.exp(0.5 * log_vars).mean()` : le σ moyen
- `decodeur_beta(moyennes)` : les μ décodés, à comparer à `X_test` ; `decodeur_beta(z_nouveau)` : les mêmes 16 codes pour chaque β

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modeles = {1 / 784: (encodeur, decodeur)}
for beta in [0, 10 / 784]:
  encodeur_beta, decodeur_beta, vae_beta = construit_vae(beta)
  vae_beta.fit(...)
  modeles[beta] = ...

_, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (beta, (encodeur_beta, decodeur_beta)) in zip(axes, sorted(modeles.items())):
  moyennes, log_vars, _ = [ops.convert_to_numpy(t) for t in encodeur_beta(X_test)]
  sigma = ...
  erreur = np.mean((ops.convert_to_numpy(decodeur_beta(...)) - X_test) ** 2)
  for etiquette, nom in enumerate(noms_classes):
    points = ...
    ax.scatter(..., s=2, alpha=0.5, label=nom)
  ax.set_title(f"β = {beta * 784:g} / 784 : σ moyen {sigma:.2f}, erreur {erreur:.4f}")
plt.show()

for beta, (_, decodeur_beta) in sorted(modeles.items()):
  print(f"β = {beta * 784:g} / 784")
  affiche_images(...)
```

### Solution

In [ ]:
modeles = {1 / 784: (encodeur, decodeur)}  # le VAE de la partie 5
for beta in [0, 10 / 784]:
  encodeur_beta, decodeur_beta, vae_beta = construit_vae(beta)
  vae_beta.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=0,
  )
  modeles[beta] = encodeur_beta, decodeur_beta

_, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (beta, (encodeur_beta, decodeur_beta)) in zip(axes, sorted(modeles.items())):
  moyennes, log_vars, _ = [ops.convert_to_numpy(t) for t in encodeur_beta(X_test)]
  sigma = np.exp(0.5 * log_vars).mean()
  # L'erreur de reconstruction des μ décodés
  erreur = np.mean((ops.convert_to_numpy(decodeur_beta(moyennes)) - X_test) ** 2)
  for etiquette, nom in enumerate(noms_classes):
    points = moyennes[y_test == etiquette]
    ax.scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, label=nom)
  ax.set_title(f"β = {beta * 784:g} / 784 : σ moyen {sigma:.2f}, erreur {erreur:.4f}")
axes[2].legend(markerscale=5, fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

for beta, (_, decodeur_beta) in sorted(modeles.items()):
  print(f"β = {beta * 784:g} / 784")
  affiche_images(ops.convert_to_numpy(decodeur_beta(z_nouveau)))

Sans KL (β = 0), le VAE retombe sur un auto-encodeur : σ tombe à 0, chaque image redevient un point, et les μ s'étalent bien au-delà de [-3, 3], en groupes séparés par des vides. Il reconstruit le mieux, mais ne génère plus : les codes tirés dans $\mathcal{N}(0, I)$ tombent tous dans la petite région autour de l'origine, et les images générées se ressemblent toutes, ou ne ressemblent à rien. Avec un β dix fois plus grand, les lois s'élargissent (σ plus grand) et les μ se resserrent en un nuage plus régulier, où les hauts se chevauchent davantage : l'erreur de reconstruction monte, le code dit moins de chaque image. Les images générées restent vraisemblables, plus lisses : avec deux dimensions, ce β reste modéré ; plus grand encore, les classes finiraient par se confondre. Le β de la partie 5 est celui de la perte d'un VAE ordinaire : la somme des erreurs des pixels, plus la KL.